# Notebook 4: PySpark MLlib Modeling
## 4 Classification Algorithms on Reddit Virality Data

**Models**:
1. Logistic Regression
2. Random Forest Classifier
3. Decision Tree Classifier
4. Gradient Boosted Trees (GBT)

All models use PySpark MLlib for distributed training on the full dataset.

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time, json
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    DecisionTreeClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Spark Session
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_Modeling')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .config('spark.driver.maxResultSize', '4g')
    .config('spark.sql.shuffle.partitions', 200)
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} | Spark UI: http://localhost:4040')

In [ ]:
# ============================================================================
# Cell 3: Load Pre-processed Train/Test Data
# ============================================================================
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

train_df = spark.read.parquet(os.path.join(PROCESSED_DIR, 'train.parquet'))
test_df = spark.read.parquet(os.path.join(PROCESSED_DIR, 'test.parquet'))

# Rename target column to 'label' for MLlib compatibility
train_df = train_df.withColumnRenamed('is_viral', 'label')
test_df = test_df.withColumnRenamed('is_viral', 'label')

# Cache for performance
train_df.cache()
test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print(f'Training set: {train_count:,} rows')
print(f'Test set:     {test_count:,} rows')
print(f'\nTraining class distribution:')
train_df.groupBy('label').count().show()

In [ ]:
# ============================================================================
# Cell 4: Define Evaluators
# ============================================================================
# Binary classification evaluator (AUC-ROC)
binary_evaluator = BinaryClassificationEvaluator(
    labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

# Multiclass evaluators for detailed metrics
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
precision_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedPrecision'
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='weightedRecall'
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)

def evaluate_model(model_name, predictions):
    """Evaluate a model and return a dictionary of metrics."""
    auc = binary_evaluator.evaluate(predictions)
    acc = accuracy_eval.evaluate(predictions)
    prec = precision_eval.evaluate(predictions)
    rec = recall_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)
    
    metrics = {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'AUC-ROC': round(auc, 4)
    }
    return metrics

print('Evaluators ready.')

## Model 1: Logistic Regression

In [ ]:
# ============================================================================
# Cell 5: Logistic Regression
# ============================================================================
print('MODEL 1: Logistic Regression')
print('=' * 60)

lr = LogisticRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0  # L2 regularization
)

start_time = time.time()
lr_model = lr.fit(train_df)
lr_train_time = time.time() - start_time

lr_predictions = lr_model.transform(test_df)
lr_metrics = evaluate_model('Logistic Regression', lr_predictions)
lr_metrics['Training Time (s)'] = round(lr_train_time, 2)

print(f'Training time: {lr_train_time:.1f}s')
for k, v in lr_metrics.items():
    print(f'  {k}: {v}')

# Training summary
lr_summary = lr_model.summary
print(f'\nObjective history (last 5): {lr_summary.objectiveHistory[-5:]}')

## Model 2: Random Forest Classifier

In [ ]:
# ============================================================================
# Cell 6: Random Forest
# ============================================================================
print('MODEL 2: Random Forest Classifier')
print('=' * 60)

rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    numTrees=100,
    maxDepth=10,
    seed=42
)

start_time = time.time()
rf_model = rf.fit(train_df)
rf_train_time = time.time() - start_time

rf_predictions = rf_model.transform(test_df)
rf_metrics = evaluate_model('Random Forest', rf_predictions)
rf_metrics['Training Time (s)'] = round(rf_train_time, 2)

print(f'Training time: {rf_train_time:.1f}s')
for k, v in rf_metrics.items():
    print(f'  {k}: {v}')

# Feature importance (top 15)
importances = rf_model.featureImportances.toArray()
print(f'\nTop 15 feature importances (index: importance):')
top_indices = np.argsort(importances)[::-1][:15]
for idx in top_indices:
    print(f'  Feature {idx}: {importances[idx]:.4f}')

## Model 3: Decision Tree Classifier

In [ ]:
# ============================================================================
# Cell 7: Decision Tree
# ============================================================================
print('MODEL 3: Decision Tree Classifier')
print('=' * 60)

dt = DecisionTreeClassifier(
    featuresCol='features',
    labelCol='label',
    maxDepth=10,
    seed=42
)

start_time = time.time()
dt_model = dt.fit(train_df)
dt_train_time = time.time() - start_time

dt_predictions = dt_model.transform(test_df)
dt_metrics = evaluate_model('Decision Tree', dt_predictions)
dt_metrics['Training Time (s)'] = round(dt_train_time, 2)

print(f'Training time: {dt_train_time:.1f}s')
for k, v in dt_metrics.items():
    print(f'  {k}: {v}')

print(f'\nTree depth: {dt_model.depth}')
print(f'Number of nodes: {dt_model.numNodes}')

## Model 4: Gradient Boosted Trees

In [ ]:
# ============================================================================
# Cell 8: Gradient Boosted Trees
# ============================================================================
print('MODEL 4: Gradient Boosted Trees (GBT)')
print('=' * 60)

gbt = GBTClassifier(
    featuresCol='features',
    labelCol='label',
    maxIter=50,
    maxDepth=5,
    seed=42
)

start_time = time.time()
gbt_model = gbt.fit(train_df)
gbt_train_time = time.time() - start_time

gbt_predictions = gbt_model.transform(test_df)
gbt_metrics = evaluate_model('Gradient Boosted Trees', gbt_predictions)
gbt_metrics['Training Time (s)'] = round(gbt_train_time, 2)

print(f'Training time: {gbt_train_time:.1f}s')
for k, v in gbt_metrics.items():
    print(f'  {k}: {v}')

print(f'\nNumber of trees: {gbt_model.getNumTrees}')
print(f'Total nodes: {gbt_model.totalNumNodes}')

## Model Comparison Summary

In [ ]:
# ============================================================================
# Cell 9: Model Comparison Table
# ============================================================================
all_metrics = [lr_metrics, rf_metrics, dt_metrics, gbt_metrics]
comparison_df = pd.DataFrame(all_metrics)

print('=' * 80)
print('MODEL COMPARISON — PySpark MLlib (Distributed)')
print('=' * 80)
print(comparison_df.to_string(index=False))

# Find best model
best_idx = comparison_df['F1-Score'].idxmax()
best_model = comparison_df.loc[best_idx, 'Model']
print(f'\n🏆 Best model by F1-Score: {best_model}')

In [ ]:
# ============================================================================
# Cell 10: Save Results and Models
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Performance metrics
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
plot_data = comparison_df[['Model'] + metrics_to_plot].melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=plot_data, x='Model', y='Score', hue='Metric', ax=axes[0], palette='viridis')
axes[0].set_title('Model Performance Comparison', fontsize=14)
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# Training time
sns.barplot(data=comparison_df, x='Model', y='Training Time (s)', ax=axes[1], palette='magma')
axes[1].set_title('Training Time Comparison', fontsize=14)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'tableau'), exist_ok=True)
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save comparison CSV for Tableau
os.makedirs(os.path.join(PROJECT_ROOT, 'tableau', 'data'), exist_ok=True)
comparison_df.to_csv(
    os.path.join(PROJECT_ROOT, 'tableau', 'data', 'spark_model_comparison.csv'),
    index=False
)
print('Model comparison saved to tableau/data/spark_model_comparison.csv')

In [ ]:
# ============================================================================
# Cell 11: Save Trained Models
# ============================================================================
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

models = {
    'logistic_regression': lr_model,
    'random_forest': rf_model,
    'decision_tree': dt_model,
    'gbt': gbt_model
}

for name, model in models.items():
    path = os.path.join(MODEL_DIR, name)
    model.write().overwrite().save(path)
    print(f'Saved: {name} -> {path}')

# Save predictions for evaluation notebook
for name, preds in [('lr', lr_predictions), ('rf', rf_predictions), 
                     ('dt', dt_predictions), ('gbt', gbt_predictions)]:
    preds.select('label', 'prediction', 'probability', 'rawPrediction') \
         .write.mode('overwrite') \
         .parquet(os.path.join(PROCESSED_DIR, f'predictions_{name}.parquet'))

print('\n✓ All models and predictions saved.')

spark.stop()
print('Spark session stopped.')